# CUDA and HIP Backward Kernels

This final notebook is meant to serve as a walkthrough as to how the other two `cupy.RawKernel` backward passes work under the hood. These kernels are ran with the assumption that $fH\times fW = sH \times sW$.

#### What Happens When we Have Overlapping Windows? 

In this scenario, downstream gradients `dinputs` would require reading from seperate windows from different receptive fields. 

We can take a $3 \times 3$ input gradient (dinputs), a $2 \times 2$ filter ($fH=2, fW=2$), and a stride of 1 ($sH=1, sW=1$).

Because the stride is 1, the sliding window steps by only 1 spatial unit, producing a $2\times 2$ upstream gradient (`dvalues`):
$$\text{dvalues} = \begin{bmatrix} \delta_{0,0} & \delta_{0,1} \\ \delta_{1,0} & \delta_{1,1} \end{bmatrix}$$

If we trace what $2\times 2$ window in `dinputs` generated each output in `dvalues`, then we get the following result. 

* **Window (0, 0)** $\rightarrow$ maps to `dvalues[0, 0]`  ($\delta_{0,0}$):
$$\begin{bmatrix} \mathbf{X} & \mathbf{X} & \cdot \\ \mathbf{X} & \mathbf{X} & \cdot \\ \cdot & \cdot & \cdot \end{bmatrix}$$

* **Window (0,1)** $\rightarrow$ maps to dvalues[0,1] ($\delta_{0,1}$):
$$\begin{bmatrix} \cdot & \mathbf{X} & \mathbf{X} \\ \cdot & \mathbf{X} & \mathbf{X} \\ \cdot & \cdot & \cdot \end{bmatrix}$$

* **Window (1,0)** $\rightarrow$ maps to dvalues[1,0] ($\delta_{1,0}$):
$$\begin{bmatrix} \cdot & \cdot & \cdot \\ \mathbf{X} & \mathbf{X} & \cdot \\ \mathbf{X} & \mathbf{X} & \cdot \end{bmatrix}$$

* **Window (1,1)** $\rightarrow$ maps to dvalues[1,1] ($\delta_{1,1}$):
$$\begin{bmatrix} \cdot & \cdot & \cdot \\ \cdot & \mathbf{X} & \mathbf{X} \\ \cdot & \mathbf{X} & \mathbf{X} \end{bmatrix}$$

#### Case A: Average Pooling

At position $(1, 1)$ inside `dinputs`, all four windows include the value. We can take a look at average pooling first:

Because coordinate $(1,1)$ is shared by all 4 windows, its final accumulated graident must sum the constributions from all 4 upstream positions.

$$\text{dinputs}[1,1] = \frac{1}{4}\delta_{0,0} + \frac{1}{4}\delta_{0,1} + \frac{1}{4}\delta_{1,0} + \frac{1}{4}\delta_{1,1}$$

the four edges require only one input from `dvalues`, while the center ones require two. 
$$\text{dinputs}[0,0] = \frac{1}{4}\delta_{0,0}$$

Inside the GPU, if we launched 4 parallel GPU threads (one for each window `i` inside `dvalues`), then all four threads will try to simultaneously try to write to `dinputs[1, 1]`. Without memory protections with atomic operations, the threads will trigger a write-collision race conditions. 

* This could also be avoided using **Reverse Thread Mapping**: Instead of mapping the GPU threads to `dvalues`, we'll map the threads to `dinputs`, with a gather sum locally before writing once. 

#### Case B: Max Pooling 
In max pooling, if supposed that position $(1,1)$ contained the max index for windows 0, 1, 2, and position $(2,2)$ contained the max index for the last window (window 3), then the correct gradients for both of these positions becomes:

$$\text{dinputs}[1,1] = \delta_{0,0} + \delta_{0,1} + \delta_{1,0}$$

$$\text{dinputs}[2,2] = \delta_{1, 1}$$

### The GPU Hardware Problem: Read-Modify-Write Race Conditions
If we attempted to iterate through the kernel like we did in the forward pass, but this time we assign one thread per `dvalues` window:

```C++
dinputs[target_idx] += gradient_value:
```
A Read-Modify-Write Race Condition occurs. Because streaming multiprocessors execute threads concurrently:

1. **Thread (0,0)** reads dinputs[1,1] (current value: 0.0).
2. **Thread (0,1)** reads dinputs[1,1] at the exact same microsecond (current value: 0.0).
3. **Thread (0,0)** computes 0.0 + 0.25 * dvalues[0,0] and writes back.
4. **Thread (0,1)** computes 0.0 + 0.25 * dvalues[0,1] and writes back, overwriting Thread (0,0)'s write.

**Atomic** or **Gather-sum** operations both aim to solve the same problem, but they benefit in distinct areas. To avoid the headache of writing multiple kernels that compile based on the dimensions of `stride` and `filter` its best we focus on a case where the windows don't overlap, maximizing performance anyways. We'll still include support for overlapping windows, but this will be found in the `_fallback` path that is calculated at compile time.

# Overview of How `_MAX_BACKWARD_NONOVERLAP_TEMPLATE` Works

In [1]:
from string import Template
_MAX_BACKWARD_NONOVERLAP_TEMPLATE = Template(r'''
$hip_include
extern "C" __global__
void $kernel_name(
    const float* __restrict__ dvalues,
    const int* __restrict__ max_indices,
    float* __restrict__ dinputs,
    const int S, const int H_pad, const int W_pad, const int C,
    const int fH, const int fW, const int sH, const int sW,
    const int H_out, const int W_out,
    const unsigned int magic_scale, const int magic_shift  
) {
    int c = blockIdx.x * blockDim.x + threadIdx.x;
    int w_out = blockIdx.y * blockDim.y + threadIdx.y;
    if (c >= C || w_out >= W_out) return;

    int h_s = blockIdx.z * blockDim.z + threadIdx.z;
    unsigned long long prod = (unsigned long long)h_s * magic_scale;
    int s = (int)(prod >> (32 + magic_shift));
    if (s >= S) return;

    int h_out = h_s - (s * H_out);

    int out_idx = ((s * H_out + h_out) * W_out + w_out) * C + c;
    int src_idx = max_indices[out_idx];
    dinputs[src_idx] = dvalues[out_idx];
}
''')

## Stage 1: Thread Grid Mapping & Boundary Guards

In the forward pass, each thread had been unraveled into its offset, used its offset to find where the starting position of the input windows location was on the input tensor, then perform the writeback operation back onto the output tensor at the very end using the `out_idx` offset.

In the backward pass, we'll still follow that same idea, but this time, because we already know that the windows do not overlap, we'll keep our gpu threads mapped to a starting value from `dvalues`. This means, **1 GPU thread maps to 1 element of `dvalues` at coordinate  $(s, h_{\text{out}}, w_{\text{out}}, c)$.
* Every `dvalues` thread looks up its target index `src_idx` and writes directly to the `dinputs[src_idx]` offset. 
* We avoid any race conditions as windows are stricly disjoint.
* Because of the lack of race conditions, we can write `dinputs[src_idx] = dvalues[out_idx]`

As a quick reminder:

* **`threadIdx.xyz` (Local Thread ID):** The offset of a **single thread** relative to the start of its own local thread block along a given axis. 
  * *Example:* If our block size along the $Z$-axis is 8 (`blockDim.z = 8`), `threadIdx.z` ranges from `0` to `7`.
* **`blockDim.xyz` (Block Dimension):** The **total number of threads per block** allocated along a given axis. It represents the chunk size assigned to each thread block.
  * *Example:* If `blockDim.z = 8`, each block processes a slice of 8 spatial/batch items. Stepping to the next block (`blockIdx.z + 1`) advances our grid coordinate by 8 threads.
* **`blockIdx.xyz` (Block ID):** The index of the **entire thread block** within the overall launch grid along a given axis. 
  * *Example:* `blockIdx.z = 0` is the first block in the grid, `blockIdx.z = 1` is the second, and so on.

The kernel will be able to map GPU thread coordinates to the tensor dimensions as follows: 

* $X$-axis: Maps to the channel $c$ (`blockIdx.x * blockDim.x + threadIdx.x)
* $Y$-axis: Maps to spatial width $w_{out}$ (`blockIdx.y * blockDim.y + threadIdx.y`)
* $Z$-axis : Holds both $s$ and $h_{out}$, (`blockIdx.z * blockDim.z + threadIdx.z`)

When we compute: 

$$\text{h\_s} = \text{blockIdx.z} \times \text{blockDim.z} + \text{threadIdx.z}$$
* `blockIdx.z * blockDim.z`: Skips paassed all preceeding blocks along the $Z$ axis to find the starting global coordinate for the current block. 
    * We move in big steps across the $(S\times H_{out})$ space. Each block is assigned a chunk (a range) of coordintates. 
* `+ threadIdx.z`: Adds the threads **local** position within that specifc block.
    * *Note:* If we **DON'T** add `threadIdx.z`, every thread inside the block will compute the exact same base $h\_s$ coordinate, causing all threads in the block to process the identical tensor index redundantly rather than working on unique elements in parallel.
    * With it, each individual thread goes through that chunk **one element at a time**, covering local positions from `0` up to `blockDim.z - 1`. 
This means, the four lines are needed
```C++
int h_s = blockIdx.z * blockDim.z + threadIdx.z;
unsigned long long prod = (unsigned long long)h_s * magic_scale;
int s = (int)(prod >> (32 + magic_shift));
if (s >= S) return;
```
The batch sample index $s = \lfloor h\_s / H_{\text{out}} \rfloor$ is needed, but we'll instead perform a scaled fixed-point fraction:
$$\frac{h\_s}{H_{\text{out}}} \approx h\_s \times \left( \frac{\text{magic\_scale}}{2^{32 + \text{magic\_shift}}} \right)$$
* `prod = (unsigned long long) h_s * magic_scale`: Multiplies two 32 bit numbers into a 64-bit integer product. 
* `prod >> (32 + magic_shift)`: Shifts right by $32 + \text{magic\_shift}$
is the binary equivalent to dividing by $2^{32 + \text{magic\_shift}}$.

Now, we have derived `s` back, and we'll use our `s` offset, to actually find the `h_{out}` offset too. This last part requires no integer divison
$$h_{\text{out}} = h\_s - (s \times H_{\text{out}})$$

## Stage 2: Flat Index Calculation & Non-Overlapping Writeback

With coordinates $(s, h_{\text{out}}, w_{\text{out}}, c)$ derived and validated against boundary guards, the kernel computes memory offsets and applies the backward gradient.

### 1. Computing `out_idx` (Row-Major Linearization)

Because tensors are stored in 1D contiguous memory arrays under an **$\text{NHWC}$ ($\text{SHWC}$)** layout, higher-dimensional indices must be mapped to a flat integer offset `out_idx`. 

The expression evaluates from left to right, multiplying each dimension index by the size of the trailing sub-tensor (its stride):

$$\text{out\_idx} = \big(\big((s \cdot H_{\text{out}} + h_{\text{out}}) \cdot W_{\text{out}} + w_{\text{out}}\big) \cdot C\big) + c$$

* **Sample Offset ($s$):** Jump past preceding batch samples. Each sample contains $H_{\text{out}} \times W_{\text{out}} \times C$ elements.
* **Height Offset ($h_{\text{out}}$):** Jump past preceding rows in the current sample. Each row contains $W_{\text{out}} \times C$ elements.
* **Width Offset ($w_{\text{out}}$):** Jump past preceding spatial columns in the current row. Each column contains $C$ channels.
* **Channel Offset ($c$):** Step directly to the target channel index (unit stride = 1).

---

### 2. Upstream Indexing & Downstream Direct Writeback

```C++
int out_idx = ((s * H_out + h_out) * W_out + w_out) * C + c;
int src_idx = max_indices[out_idx];
dinputs[src_idx] = dvalues[out_idx];
```

* `max_indices[out_idx]`: During the forward pass, max-pooling recorded the exact global input offset (`best_idx`) for the maximum activation for window `out_idx`. We save that directly to the updated offset
* `dinputs[src_idx] = dvalues[out_idx]`: The non-overlapping assumption means that each trhead writes to a unique `src_idx`. Thus, this concludes an overview of the first backward pass kernel.

# Overview of How `_AVG_BACKWARD_NONOVERLAP_TEMPLATE` Works

In [2]:
_AVG_BACKWARD_NONOVERLAP_TEMPLATE = Template(r'''
$hip_include
extern "C" __global__
void $kernel_name(
    const float* __restrict__ dvalues,
    float* __restrict__ dinputs,
    const int S, const int H_pad, const int W_pad, const int C,
    const int fH, const int fW, const int sH, const int sW,
    const int H_out, const int W_out,
    const unsigned int magic_scale, const int magic_shift  
) {
    int c = blockIdx.x * blockDim.x + threadIdx.x;
    int w_out = blockIdx.y * blockDim.y + threadIdx.y;
    if (c >= C || w_out >= W_out) return;

    int h_s = blockIdx.z * blockDim.z + threadIdx.z;
    unsigned long long prod = (unsigned long long)h_s * magic_scale;
    int s = (int)(prod >> (32 + magic_shift));
    if (s >= S) return;

    int h_out = h_s - (s * H_out);

    int h_start = h_out * sH;
    int w_start = w_out * sW;
    int batch_offset = s * H_pad * W_pad * C;

    int out_idx = ((s * H_out + h_out) * W_out + w_out) * C + c;
    float avg_val = dvalues[out_idx] / (float)(fH * fW);

    for (int fh = 0; fh < fH; ++fh) {
        int h_in = h_start + fh;
        int row_offset = batch_offset + (h_in * W_pad) * C;

        for (int fw = 0; fw < fW; ++fw) {
            int w_in = w_start + fw;
            int in_idx = row_offset + w_in * C + c;
            dinputs[in_idx] = avg_val;
        }
    }
}
''')

## Stage 2 : Receptive Field Mapping & Spatial Scatter

Stage 1 is skipped as the process is the same for both max and average pooling, as they both assume non-overlapping windows. 

In Average Pooling, every element in the forward filter window contributed equally with a weight of $\frac{1}{fH \times fW}$. 

During the backward pass, each thread calculates this uniform derivative term once, then **scatters (writes)** it across its entire $fH \times fW$ receptive field in `dinputs`.

```C++
int h_start = h_out * sH;
int w_start = w_out * sW;
int batch_offset = s * H_pad * W_pad * C;
```
To map where the scaler graident should be scattered in `dinputs`:
* `h_start` & `w_start`: Set the output spatial coordinates back to the top-left corder of the corresponding window on the input grid. 
* `batch_offset`: Precomputes the stride memory offset to jump a batch sample inside `dinputs`. 


## Stage 3: Nested Loop Scatter Writeback

In [ ]:
"""
int out_idx = ((s * H_out + h_out) * W_out + w_out) * C + c;
float avg_val = dvalues[out_idx] / (float)(fH * fW);

for (int fh = 0; fh < fH; ++fh) {
    int h_in = h_start + fh;
    int row_offset = batch_offset + (h_in * W_pad) * C;

    for (int fw = 0; fw < fW; ++fw) {
        int w_in = w_start + fw;
        int in_idx = row_offset + w_in * C + c;
        dinputs[in_idx] = avg_val;
    }
}
"""

* `h_in = h_start + fh`: Given the temporary placement inside the filter height, and the local starting position given `h_out * sH`, we'll calculate where the **exact row position** inside the input tensor that the current thread is scattering its gradients to. 
* `row_offset`: We follow the same logic as we did in the forward pass, we'll compute the offset in the outer loop and measure the offest of `dinputs` based on the equation:
$$\text{row\_offset} = \text{batch\_offset} + (h_{\text{in}} \times W_{\text{pad}}) \times C$$

Now for the inner loop, for each row, track over the filter width dimensions or the columns of the image. 

* `int w_in = w_start + fw`: This denotes our temporary placement inside the filter width window. In other words, it combines the exact column position inside `dinputs` by combinging the left edge origin `w_start` with the local filter window offset `fw`. 
* `int in_idx = row_offset + w_in * C + c`: Our actual index offset of the input tensor `dinputs`. This is based on the row offset which holds the batch offset along with offset of the row within the batch, as well as the width offset along with th eoffset of the channel C.
* `dinputs[in_idx] = avg_val`: Finally, this section will rewrite the value at with the averaged gradient.
Our threads update $fH \times fW$ elements inside the `dinputs` tensor before retiring. 

